<a href="https://colab.research.google.com/github/minbj1226/computer_vision-study/blob/main/CNN_Architectures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN 모델
- 모델의 종류 및 각각에 대한 등장 배경, 전체 구조, 핵심 아이디어 정리

## LeNet-5
- 등장 배경: 기존의 패턴 인식에서 이용되는 전통적인 모델이 hand-designed feature extractor로 특징을 추출하고, fully-connected multi-layer networks를 분류기로 사용 -> 제한된 특징만 추출, 너무 많은 매개변수를 포함
- 전체 구조: Conv(C1) - AvgPool - Conv(C2) - AvgPool - Conv(C3) - FC - FC - FC
- 핵심 아이디어: CNN 기본 틀 제시

## AlexNet
- 등장 배경: ImageNet과 같은 대규모 이미지 분류 문제에서 기존 방법보다 높은 성능을 내기 위해 제안되었으며, 딥 CNN이 GPU 학습과 ReLU, dropout, data augmentation을 통해 실전에서 효과적임을 보여준 모델
- 전체 구조: Conv(C1) - MaxPool - Conv(C2) - MaxPool - Conv(C3) - Conv(C4) - Conv(C5) - MaxPool - FC - FC - FC
- 핵심 아이디어: ReLU 활성화 함수를 사용해 학습 속도를 높였고, Max Pooling과 dropout, data augmentation를 통해 대규모 이미지 분류에서 높은 성능

## VGG
- 등장 배경: 네트워크를 더 깊게 쌓는 것이 성능 향상에 어떤 영향을 주는지 탐구하고, 작은 3x3 convolution을 반복적으로 사용해 단순하고 깊은 구조를 만들기 위해 제안된 모델
- 전체 구조: [Conv(64) - Conv(64)] - MaxPool - [Conv(128) - Conv(128)] - MaxPool - [Conv(256) - Conv(256) - Conv(256)] - MaxPool - [Conv(512) - Conv(512) - Conv(512)] - MaxPool - [Conv(512) - Conv(512) - Conv(512)] - MaxPool - FC - FC - FC - Softmax
- 핵심 아이디어: 큰 커널 대신에 작은 3x3 convolution을 여러 층 쌓아 같은 receptive field를 더 적은 파라미터로 표현

## Inception
- 등장 배경: 더 깊고 넓은 네트워크가 성능은 좋지만, 계산량과 파라미터 수가 너무 커지는 문제를 해결하기 위해 제안된 모델
- 전체 구조:
  - branch 1: 1*1 Conv
  - branch 2: 1*1 Conv -> 3 * 3 Conv
  - branch 3: 1*1 Conv -> 5 * 5 Conv
  - branch 4: max pooling -> 1 * 1 Conv
- 핵심 아이디어:
  - 여러 크기의 receptive field를 동시에 본다
  - 1*1 convolution으로 채널 수를 줄여 계산량을 감소
  - 병렬 branch 결과를 concat

## ResNet
- 등장 배경: 네트워크가 깊어질수록 기울기 소실과 최적화 어려움으로 인해 학습이 잘 되지 않고, 층을 더 쌓아도 training error가 오히려 증가하는 degradation problem을 해결하기 위해 제안된 모델
- 전체 구조: 입력 x를 main path와 shortcut path로 나눈 뒤, main path에서는 convolution 연산을 수행하고 shortcut path에서는 입력을 그대로 전달한 후, 두 경로의 출력을 더해 residual block을 구성하는 구조
- 핵심 아이디어:입력 x를 shortcut(skip connection)으로 뒤쪽 층에 직접 더해줘, 네트워크가 원래 함수 H(x) 대신 잔차 F(x)=H(x)-x를 학습하도록 만든 구조

## MobileNet v1
- 등장 배경: 모바일 및 임베디드 환경에서 이전 모델들은 크기가 크고 연산량이 많아 사용하기 어려움
- 전체 구조: 기존 convolution 이후, depthwise convolution과 pointwise convolution으로 이루어진 depthwise separable convolution block을 반복적으로 쌓는 구조
- 핵심 아이디어: 일반 convolution을 depthwise convolution과 pointwise convolution으로 분리하여, depthwise는 각 채널별로 공간 특징을 추출하고 pointwise는 1x1 convolution으로 채널 간 정보를 결합함으로써 연산량과 파라미터 수를 크게 줄이는 것

In [1]:
# LeNet 구조
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 6, 5)
    self.conv2 = nn.Conv2d(6, 16, 5)
    self.fc1 = nn.Linear(16*5*5, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, x):
    x = F.avg_pool2d(F.tanh(self.conv1(x)), 2)
    x = F.avg_pool2d(F.tanh(self.conv2(x)), 2)
    x = torch.flatten(x, 1)
    x = F.tanh(self.fc1(x))
    x = F.tanh(self.fc2(x))
    x = self.fc3(x)

    return x

In [2]:
# AlexNet 구조
import torch
import torch.nn as nn
import torch.nn.functional as F

class AlexNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, 96, 11, stride=4)
    self.conv2 = nn.Conv2d(96, 256, 5, padding=2)
    self.conv3 = nn.Conv2d(256, 384, 3, padding=1)
    self.conv4 = nn.Conv2d(384, 384, 3, padding=1)
    self.conv5 = nn.Conv2d(384, 256, 3, padding=1)

    self.fc1 = nn.Linear(6 * 6 * 256, 4096)
    self.fc2 = nn.Linear(4096, 4096)
    self.fc3 = nn.Linear(4096, 1000)

  def forward(self, x):
    x = F.relu(F.max_pool2d(self.conv1(x), kernel_size=3, stride=2))
    x = F.relu(F.max_pool2d(self.conv2(x), kernel_size=3, stride=2))
    x = F.relu(self.conv3(x))
    x = F.relu(self.conv4(x))
    x = F.relu(F.max_pool2d(self.conv5(x), kernel_size=3, stride=2))

    x = torch.flatten(x, 1)
    x = F.dropout(F.relu(self.fc1(x)), p=0.5, training=self.training)
    x = F.dropout(F.relu(self.fc2(x)), p=0.5, training=self.training)
    x = self.fc3(x)

    return x

In [3]:
# VGG-16

class VGG16(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
    self.conv2 = nn.Conv2d(64, 64, 3, padding=1)
    self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
    self.conv4 = nn.Conv2d(128, 128, 3, padding=1)
    self.conv5 = nn.Conv2d(128, 256, 3, padding=1)
    self.conv6 = nn.Conv2d(256, 256, 3, padding=1)
    self.conv7 = nn.Conv2d(256, 256, 3, padding=1)
    self.conv8 = nn.Conv2d(256, 512, 3, padding=1)
    self.conv9 = nn.Conv2d(512, 512, 3, padding=1)
    self.conv10 = nn.Conv2d(512, 512, 3, padding=1)
    self.conv11 = nn.Conv2d(512, 512, 3, padding=1)
    self.conv12 = nn.Conv2d(512, 512, 3, padding=1)
    self.conv13 = nn.Conv2d(512, 512, 3, padding=1)

    self.fc1 = nn.Linear(512 * 7 * 7, 4096)
    self.fc2 = nn.Linear(4096, 4096)
    self.fc3 = nn.Linear(4096, 1000)

  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = F.relu(self.conv2(x))
    x = F.max_pool2d(x, kernel_size=2, stride=2)
    x = F.relu(self.conv3(x))
    x = F.relu(self.conv4(x))
    x = F.max_pool2d(x, kernel_size=2, stride=2)
    x = F.relu(self.conv5(x))
    x = F.relu(self.conv6(x))
    x = F.relu(self.conv7(x))
    x = F.max_pool2d(x, kernel_size=2, stride=2)
    x = F.relu(self.conv8(x))
    x = F.relu(self.conv9(x))
    x = F.relu(self.conv10(x))
    x = F.max_pool2d(x, kernel_size=2, stride=2)
    x = F.relu(self.conv11(x))
    x = F.relu(self.conv12(x))
    x = F.relu(self.conv13(x))
    x = F.max_pool2d(x, kernel_size=2, stride=2)

    x = torch.flatten(x, 1)
    x = F.dropout(F.relu(self.fc1(x)), p=0.5, training=self.training)
    x = F.dropout(F.relu(self.fc2(x)), p=0.5, training=self.training)
    x = self.fc3(x)

    return x

In [4]:
# Inception
# 여러 Inception block을 쌓아 전체 Inception 계열 네트워크를 구성할 수 있다.

import torch
import torch.nn as nn
import torch.nn.functional as F

class InceptionBlock(nn.Module):
  def __init__(self):
    super().__init__()

    self.branch1 = nn.Conv2d(192, 64, 1)

    self.branch2 = nn.Sequential(
        nn.Conv2d(192, 96, 1),
        nn.ReLU(inplace=True),
        nn.Conv2d(96, 128, 3, padding=1)
    )

    self.branch3 = nn.Sequential(
        nn.Conv2d(192, 16, 1),
        nn.ReLU(inplace=True),
        nn.Conv2d(16, 32, 5, padding=2)
    )

    self.branch4 = nn.Sequential(
        nn.MaxPool2d(3, stride=1, padding=1),
        nn.Conv2d(192, 32, 1)
    )

    self.relu = nn.ReLU(inplace=True)

  def forward(self, x):
    b1 = self.relu(self.branch1(x))
    b2 = self.relu(self.branch2(x))
    b3 = self.relu(self.branch3(x))
    b4 = self.relu(self.branch4(x))

    return torch.cat([b1, b2, b3, b4], dim=1)

In [5]:
# ResNet 간단한 설계
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResNet(nn.Module):
  def __init__(self, n_chans1=32):
    super().__init__()
    self.n_chans1 = n_chans1
    self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(n_chans1 // 2, n_chans1 // 2, kernel_size=3, padding=1)
    self.fc1 = nn.Linear(4 * 4 * n_chans1 // 2, 32)
    self.fc2 = nn.Linear(32, 2)

  def forward(self, x):
    out = F.max_pool2d(torch.relu(self.conv1(x)), 2)
    out = F.max_pool2d(torch.relu(self.conv2(out)), 2)
    out1 = out
    out = F.max_pool2d(torch.relu(self.conv3(out)) + out1, 2)
    out = torch.flatten(out, 1)
    out = torch.relu(self.fc1(out))
    out = self.fc2(out)

    return out

In [6]:
# MobileNet
import torch
import torch.nn as nn
import torch.nn.functional as F

class MobileNet(nn.Module):
  def __init__(self, ch_in, ch_out):
    super().__init__()
    self.depthwise = nn.Conv2d(
        in_channels = ch_in,
        out_channels = ch_out,
        kernel_size = 3,
        stride = 1,
        padding = 1,
        bias = False
    )

    self.pointwise = nn.Conv2d(
        in_channels = ch_in,
        out_channels = ch_out,
        kernel_size = 1,
        stride = 1,
        padding = 0,
        bias=False
    )

    self.bn1 = nn.BatchNorm2d(ch_in)
    self.bn2 = nn.BatchNorm2d(ch_out)
    self.relu = nn.ReLU(inplace=True)

  def forward(self, x):
    x = self.depthwise(x)
    x = self.bn1(x)
    x = self.relu(x)

    x = self.pointwise(x)
    x = self.bn2(x)
    x = self.relu(x)
    return x